In [ ]:
!pip install -q \
speechbrain==0.5.16 \
faster-whisper==1.0.3 \
ctranslate2==4.4.0 \
huggingface_hub==0.21.4 \
librosa==0.10.1 \
soundfile \
scikit-learn \
pandas==2.2.2

In [ ]:
import numpy as np
import pandas as pd
import librosa
import traceback
from datetime import timedelta

from faster_whisper import WhisperModel
from speechbrain.pretrained import EncoderClassifier
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity

import torch
import os


In [ ]:
whisper_model = WhisperModel("base", compute_type="int8")

embedding_model = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb",
    run_opts={"device": "cpu"}
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_token.py:88: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
def convert_time(secs):
    return str(timedelta(seconds=secs))

In [ ]:
def transcribe_audio(audio_file, language="en", beam_size=5, best_of=5):
    segments, _ = whisper_model.transcribe(
        audio_file,
        language=language,
        beam_size=beam_size,
        best_of=best_of
    )

    results = []
    for seg in segments:
        results.append({
            "start": seg.start,
            "end": seg.end,
            "text": seg.text
        })

    return results

In [ ]:
def extract_segment_embedding(y, sr, segment, total_duration, embedding_model):
    try:
        start = segment["start"]
        end = min(segment["end"], total_duration)

        start_sample = int(start * sr)
        end_sample = int(end * sr)

        waveform = y[start_sample:end_sample]

        waveform = torch.tensor(waveform).unsqueeze(0)

        with torch.no_grad():
            embedding = embedding_model.encode_batch(waveform)
            embedding = embedding / torch.norm(embedding, dim=-1, keepdim=True)

        return embedding.squeeze().cpu().numpy()

    except Exception as e:
        print("Error:", e)
        return None

In [ ]:
def compute_segment_embeddings(audio_file, segments, embedding_model):
    y, sr = librosa.load(audio_file, sr=16000, mono=True)
    duration = librosa.get_duration(y=y, sr=sr)

    embeddings = []
    valid_segments = []

    for seg in segments:
        emb = extract_segment_embedding(y, sr, seg, duration, embedding_model)
        if emb is not None:
            embeddings.append(emb)
            valid_segments.append(seg)

    return np.vstack(embeddings), duration, valid_segments

In [ ]:
def cluster_embeddings(embeddings, n_clusters):
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    kmeans.fit(embeddings)

    return kmeans.labels_, kmeans.cluster_centers_

In [ ]:
def compute_cluster_averages(embeddings, labels, n_clusters):
    cluster_avg = {}

    for i in range(n_clusters):
        cluster_embs = embeddings[labels == i]
        avg = np.mean(cluster_embs, axis=0)
        avg = avg / np.linalg.norm(avg)   # 🔥 IMPORTANT FIX
        cluster_avg[i] = avg

    return cluster_avg

In [ ]:
def load_known_speaker_embeddings(files, embedding_model):
    known = {}

    for f in files:
        label = os.path.splitext(os.path.basename(f))[0]

        y, sr = librosa.load(f, sr=16000, mono=True)

        waveform = torch.tensor(y).unsqueeze(0)

        with torch.no_grad():
            emb = embedding_model.encode_batch(waveform)
            emb = emb / torch.norm(emb, dim=-1, keepdim=True)
            emb = emb.mean(dim=1)

        known[label] = emb.squeeze().cpu().numpy()

    return known

In [ ]:
def assign_speaker_labels(cluster_avg_embeddings, known_speaker_embeddings, threshold=0.7):
    mapping = {}
    best_scores = {}

    for cid, avg_emb in cluster_avg_embeddings.items():
        best_label = None
        best_score = -1

        for name, known_emb in known_speaker_embeddings.items():
            score = cosine_similarity(
                avg_emb.reshape(1, -1),
                known_emb.reshape(1, -1)
            )[0][0]

            if score > best_score:
                best_score = score
                best_label = name

        best_scores[cid] = best_score

        if best_score >= threshold:
            mapping[cid] = best_label
        else:
            mapping[cid] = "Unknown"

    return mapping, best_scores

In [ ]:
def run_pipeline(audio_file, embedding_model, known_embeddings,
                 n_clusters=3, threshold=0.7):

    print(f"\nProcessing {audio_file}")

    segments = transcribe_audio(audio_file)

    embeddings, duration, valid_segments = compute_segment_embeddings(
        audio_file, segments, embedding_model
    )

    labels, centroids = cluster_embeddings(embeddings, n_clusters)

    cluster_avg = compute_cluster_averages(embeddings, labels, n_clusters)

    speaker_map, best_scores = assign_speaker_labels(
        cluster_avg, known_embeddings, threshold
    )

    for i, seg in enumerate(valid_segments):
        seg["cluster"] = labels[i]
        seg["speaker"] = speaker_map[labels[i]]

    df = pd.DataFrame(valid_segments)

    df["Start"] = df["start"].apply(convert_time)
    df["End"] = df["end"].apply(convert_time)

    return df, len(valid_segments), centroids, best_scores

In [ ]:
!wget -O wavs.tar.gz "https://drive.google.com/uc?id=1ZcSTNF43seMPo0nF8sdXY5922Luc2bqn"
!tar -xvf wavs.tar.gz

--2026-03-19 16:12:45--  https://drive.google.com/uc?id=1ZcSTNF43seMPo0nF8sdXY5922Luc2bqn
Resolving drive.google.com (drive.google.com)... 142.250.125.100, 142.250.125.102, 142.250.125.101, ...
Connecting to drive.google.com (drive.google.com)|142.250.125.100|:443... connected.
HTTP request sent, awaiting response... 303 See Other
Location: https://drive.usercontent.google.com/download?id=1ZcSTNF43seMPo0nF8sdXY5922Luc2bqn [following]
--2026-03-19 16:12:45--  https://drive.usercontent.google.com/download?id=1ZcSTNF43seMPo0nF8sdXY5922Luc2bqn
Resolving drive.usercontent.google.com (drive.usercontent.google.com)... 173.194.194.132, 2607:f8b0:4001:c68::84
Connecting to drive.usercontent.google.com (drive.usercontent.google.com)|173.194.194.132|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3553280 (3.4M) [application/octet-stream]
Saving to: ‘wavs.tar.gz’

wavs.tar.gz         100%[===================>]   3.39M  --.-KB/s    in 0.06s   

2026-03-19 16:12:46 (53.8 MB

In [ ]:
known_files = [
    "speaker_A.wav",
    "speaker_B.wav",
    "speaker_C.wav",
    "speaker_D.wav",
    "speaker_E.wav"
]

known_embeddings = load_known_speaker_embeddings(known_files, embedding_model)

In [ ]:
df_clean, n_clean, cent_clean, scores_clean = run_pipeline(
    "sample.wav", embedding_model, known_embeddings
)

df_noisy, n_noisy, cent_noisy, scores_noisy = run_pipeline(
    "sample_noisy.wav", embedding_model, known_embeddings
)


Processing sample.wav

Processing sample_noisy.wav


In [ ]:
df_clean, n_clean, cent_clean, scores_clean

(   start    end                                               text  cluster  \
 0   0.00   9.54   This device picked up the same amount of masc...        2   
 1   9.54  15.20   likely produce similar increases in the cost ...        2   
 2  15.20  18.52                                Do you doubt Homer?        0   
 3  18.52  25.12   One can also build cords by stacking fifths y...        2   
 4  25.12  31.12   Forget Apollo, he said, with his suggestion o...        0   
 5  31.12  38.12   Myconical bots with cylindrical necks are esp...        1   
 
      speaker           Start             End  
 0  speaker_C         0:00:00  0:00:09.540000  
 1  speaker_C  0:00:09.540000  0:00:15.200000  
 2    Unknown  0:00:15.200000  0:00:18.520000  
 3  speaker_C  0:00:18.520000  0:00:25.120000  
 4    Unknown  0:00:25.120000  0:00:31.120000  
 5    Unknown  0:00:31.120000  0:00:38.120000  ,
 6,
 array([[-8.50184411e-02,  1.93376075e-02, -6.41290098e-03,
          1.16918851e-02,  4.97280993

In [ ]:
print(scores_clean)

{0: np.float32(0.599483), 1: np.float32(0.45877), 2: np.float32(0.75754815)}


In [ ]:
print(df_clean)

   start    end                                               text  cluster  \
0   0.00   9.54   This device picked up the same amount of masc...        2   
1   9.54  15.20   likely produce similar increases in the cost ...        2   
2  15.20  18.52                                Do you doubt Homer?        0   
3  18.52  25.12   One can also build cords by stacking fifths y...        2   
4  25.12  31.12   Forget Apollo, he said, with his suggestion o...        0   
5  31.12  38.12   Myconical bots with cylindrical necks are esp...        1   

     speaker           Start             End  
0  speaker_C         0:00:00  0:00:09.540000  
1  speaker_C  0:00:09.540000  0:00:15.200000  
2    Unknown  0:00:15.200000  0:00:18.520000  
3  speaker_C  0:00:18.520000  0:00:25.120000  
4    Unknown  0:00:25.120000  0:00:31.120000  
5    Unknown  0:00:31.120000  0:00:38.120000  


In [ ]:
print(n_clean)

6


In [ ]:
print(cent_clean)

[[-8.50184411e-02  1.93376075e-02 -6.41290098e-03  1.16918851e-02
   4.97280993e-02 -3.86588648e-02  1.73190273e-02  9.10267830e-02
  -3.62230130e-02 -4.70348746e-02  3.55604105e-03  5.54082282e-02
   4.88036014e-02  4.55714390e-03  1.10005885e-02  8.40174034e-02
  -8.68529677e-02  6.62404522e-02 -6.98854476e-02  2.59866156e-02
   5.70799410e-02 -2.09616460e-02 -5.32918572e-02  9.89902169e-02
  -6.60703853e-02  4.06967625e-02 -1.09532978e-02  4.88763079e-02
   2.90837716e-02 -8.38851705e-02 -2.13464648e-02 -5.90473115e-02
  -1.02991506e-01 -6.42975867e-02 -2.68327836e-02 -7.71129951e-02
   3.30267549e-02 -9.92857218e-02 -4.22827378e-02  5.58143258e-02
  -6.19565882e-02 -5.19588254e-02 -9.40217003e-02 -4.51546013e-02
   2.17950623e-02 -1.46519750e-01  1.40632093e-01 -4.82332781e-02
  -2.35054940e-02  1.22712879e-02 -7.68491179e-02  4.06534560e-02
   1.44163668e-02  6.23222254e-02  5.79307824e-02 -7.67384022e-02
   3.40350457e-02  1.25545617e-02 -5.70110232e-02  1.11217313e-02
   1.00529

In [ ]:
avg_score = round(sum(scores_clean.values()) / len(scores_clean), 3)
print(avg_score)

0.605


In [ ]:
print(scores_noisy)

{0: np.float32(0.5872623), 1: np.float32(0.5447318), 2: np.float32(0.66841805)}


In [ ]:
avg_score_noisy = round(sum(scores_noisy.values()) / len(scores_noisy), 3)
print(avg_score_noisy)

0.6


In [ ]:
print(n_noisy)

8


In [ ]:
percent_increase = round(((n_noisy - n_clean) / n_clean) * 100, 2)
print(percent_increase)

33.33


In [ ]:
total_duration = (df_clean["end"] - df_clean["start"]).sum()
print(round(total_duration, 2))

38.12


In [ ]:
df_clean

,start,end,text,cluster,speaker,Start,End
0,0.00,9.54,This device picked up the same amount of masc...,2,speaker_C,0:00:00,0:00:09.540000
1,9.54,15.20,likely produce similar increases in the cost ...,2,speaker_C,0:00:09.540000,0:00:15.200000
2,15.20,18.52,Do you doubt Homer?,0,Unknown,0:00:15.200000,0:00:18.520000
3,18.52,25.12,One can also build cords by stacking fifths y...,2,speaker_C,0:00:18.520000,0:00:25.120000
4,25.12,31.12,"Forget Apollo, he said, with his suggestion o...",0,Unknown,0:00:25.120000,0:00:31.120000
5,31.12,38.12,Myconical bots with cylindrical necks are esp...,1,Unknown,0:00:31.120000,0:00:38.120000
